# B2 — Spam Classification: KNN vs SVM

**Laboratory Practice III (410246), Group B**

Dataset reference: https://www.kaggle.com/uciml/sms-spam-collection-dataset

Run cells top-to-bottom. Set `CSV_PATH` to your downloaded CSV file to use real data; `None` runs the included synthetic demonstration. Results on synthetic data are not real-world model performance.

In [ ]:
# Set to a local CSV filename/path after downloading the Kaggle dataset.
# Keep None to run with built-in synthetic demonstration data.
CSV_PATH = None  # Example: 'spam.csv'

## Implementation and evaluation

In [ ]:
"""B2: Compare KNN and SVM spam classification using TF-IDF features."""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

def main():
    if CSV_PATH:  # Load real email/SMS CSV here; supports v1/v2 or label/text columns.
        df=pd.read_csv(CSV_PATH,encoding='latin-1')
    else:
        spam=['Claim your cash prize now','You won a free phone','Urgent lottery winner call us',
              'Exclusive discount click here','Get rich fast offer','Free voucher redeem today',
              'Winner selected for reward','Claim bonus money instantly']
        ham=['Can we meet for lunch','Please send the report','Are you home tonight',
             'Meeting at nine tomorrow','Thanks for the update','Remember to buy groceries',
             'Please review my assignment','See you at the office']
        df=pd.DataFrame([(s+' '+str(i),'spam') for i in range(8) for s in spam]+[(s+' '+str(i),'ham') for i in range(8) for s in ham],columns=['text','label'])
        print('Using synthetic demonstration messages.')
    df=df.rename(columns={'v1':'label','v2':'text','Category':'label','Message':'text'})
    if not {'label','text'}<=set(df):raise ValueError('Expected label/text or v1/v2 columns')
    df=df[['text','label']].dropna();df['label']=df.label.astype(str).str.strip().str.lower()
    df=df[df.label.isin(['spam','ham'])]
    if df.label.nunique()!=2 or df.label.value_counts().min()<2:raise ValueError('Need >=2 examples of each class')
    Xtr,Xte,ytr,yte=train_test_split(df.text.astype(str),df.label,test_size=.25,stratify=df.label,random_state=42)
    for name,model in [('KNN',KNeighborsClassifier(n_neighbors=min(5,len(Xtr)))),('SVM',LinearSVC(random_state=42))]:
        pipe=Pipeline([('tfidf',TfidfVectorizer(stop_words='english')),('classifier',model)])
        pipe.fit(Xtr,ytr);pred=pipe.predict(Xte)
        print('\n',name,'Accuracy:',round(accuracy_score(yte,pred),4))
        print('Confusion matrix [ham, spam]:\n',confusion_matrix(yte,pred,labels=['ham','spam']))
        print(classification_report(yte,pred,zero_division=0))
main()
